# Lab 01 — Prompt engineering and structured outputs

**Level:** Beginner · **Duration:** ~75 minutes · **Covers:** H1 prompt optimisation lab (45 min) · H2 structured output generation (30 min)

Lab guide: [`../lab-01-prompt-engineering-structured-outputs.md`](../lab-01-prompt-engineering-structured-outputs.md) — read each part there first; this notebook holds the code.

This lab is guided: every cell is given. You run it, observe, and record what you see. Model output varies from run to run, so record **your** results, not the guide's.

In [1]:
# (notebook) Setup: run this cell first. It loads the repo-root .env and creates both SDK clients.
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import anthropic
import openai
import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env
pd.set_option("display.max_colwidth", 110)

# labs/module-02 holds data/ and templates/; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-02" and (p / "data").is_dir())
DATA_DIR = LAB_DIR / "data"

# Model IDs come from .env, never from code (labs/setup/.env.example has the defaults).
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "").strip()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "").strip()

# Each SDK reads its key from the environment; a client exists only when key and model are set.
openai_client = openai.OpenAI() if os.getenv("OPENAI_API_KEY") and OPENAI_MODEL else None
claude_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") and ANTHROPIC_MODEL else None
PROVIDERS = [p for p, c in (("openai", openai_client), ("anthropic", claude_client)) if c is not None]


def claude_text(message) -> str:
    """Join the text blocks of a Claude response; thinking blocks are skipped."""
    return "".join(block.text for block in message.content if block.type == "text")


def run_parallel(fn, jobs: list[tuple], max_workers: int = 8) -> list:
    """Call fn(*job) for every job on a small thread pool; results come back in job order."""
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        return list(pool.map(lambda job: fn(*job), jobs))


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(f"Lab folder : {LAB_DIR}")
print(f"OpenAI     : {'✅ ' + OPENAI_MODEL if openai_client else '❌ OPENAI_API_KEY or OPENAI_MODEL missing'}")
print(f"Claude     : {'✅ ' + ANTHROPIC_MODEL if claude_client else '❌ ANTHROPIC_API_KEY or ANTHROPIC_MODEL missing'}")
if not PROVIDERS:
    print(
        "\n❌ No provider is configured. Copy labs/setup/.env.example to the repo-root .env, add your keys,\n"
        "   restart the kernel and run this cell again. Until then, cells that call a model are skipped."
    )

Lab folder : /Users/saravana/AgenticAI/agentic-ai-python/labs/module-02
OpenAI     : ✅ gpt-5-mini
Claude     : ✅ claude-haiku-5-5


## Part 1 — The labelled ticket set and a baseline to beat (10 min)

You cannot improve a prompt you do not measure. Load the 30 labelled tickets, read the policy
they were labelled with, and score a keyword baseline: every prompt in this lab has to beat it.

In [2]:
# (notebook) 6 worked examples (for few-shot prompts) and 24 test tickets, labelled under POL-OPS-004.
tickets = pd.read_csv(DATA_DIR / "triage_tickets.csv")
EXAMPLES = tickets[tickets.split == "example"].reset_index(drop=True)
TEST = tickets[tickets.split == "test"].reset_index(drop=True)
POLICY = (DATA_DIR / "triage_policy.md").read_text(encoding="utf-8")

print(f"{len(EXAMPLES)} examples · {len(TEST)} test tickets · languages {TEST.language.value_counts().to_dict()}")
print("priorities:", TEST.priority.value_counts().sort_index().to_dict())
TEST[["ticket_id", "text", "category", "priority", "assignment_group"]].head(6)

6 examples · 24 test tickets · languages {'en': 22, 'fr': 1, 'es': 1}
priorities: {'P1': 4, 'P2': 5, 'P3': 6, 'P4': 9}


,ticket_id,text,category,priority,assignment_group
0,INC-104411,My account is locked after I mistyped my password a few times. I need to log in for a customer call in 20 ...,access,P3,Service Desk
1,INC-104412,MFA push notifications stopped arriving on my new phone. I can still use the SMS code fallback.,access,P4,Service Desk
2,INC-104413,Nobody in the APAC region can sign in to anything since 02:00 UTC. The sign-in page says 'identity provide...,access,P1,Platform Engineering
3,INC-104414,The print queue for printer PR-3F-07 is stuck and jobs just sit there. I can print to the printer next to ...,endpoint,P4,Service Desk
4,INC-104415,Mon ordinateur portable affiche l'écran de récupération du chiffrement au démarrage et demande une clé. Je...,endpoint,P3,Service Desk
5,INC-104416,My mail client says messages have been stuck in my outbox since yesterday. Receiving works.,email_collab,P4,Service Desk


In [3]:
# (notebook) The label contract and the scorer.
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

Category = Literal["access", "network", "endpoint", "email_collab", "business_app", "platform", "security"]
Priority = Literal["P1", "P2", "P3", "P4"]
Group = Literal["Service Desk", "Network Ops", "Finance", "Platform Engineering", "Security Operations"]
FIELDS = ["category", "priority", "assignment_group"]


class TriageLabel(BaseModel):
    category: Category
    priority: Priority
    assignment_group: Group


def score(predictions: pd.DataFrame) -> dict[str, float]:
    """Accuracy per field, and for all three at once, against the gold labels of TEST."""
    merged = TEST.merge(predictions[["ticket_id", *FIELDS]], on="ticket_id", suffixes=("", "_pred"), how="left")
    hits = pd.DataFrame({f: merged[f] == merged[f + "_pred"] for f in FIELDS})
    return {**{f: round(float(hits[f].mean()), 3) for f in FIELDS}, "all three": round(float(hits.all(axis=1).mean()), 3)}

In [4]:
# (notebook) A keyword baseline: ten lines of rules, no model. Every prompt below must beat it.
KEYWORDS = {
    "security": ["phishing", "malware", "trojan", "antivirus", "clicked", "lost", "stolen", "suspicious"],
    "network": ["vpn", "wi-fi", "wifi", "network"],
    "business_app": ["erp", "invoice", "payment", "journal", "zahlungslauf"],
    "platform": ["server", "storage", "build", "pipeline", "cluster"],
    "email_collab": ["email", "mail", "outbox", "calendar", "out-of-office"],
    "access": ["password", "account", "locked", "mfa", "sign in", "access"],
}
ROUTING = {"access": "Service Desk", "endpoint": "Service Desk", "email_collab": "Service Desk", "network": "Network Ops",
           "business_app": "Finance", "platform": "Platform Engineering", "security": "Security Operations"}


def keyword_baseline(text: str) -> dict:
    t = text.lower()
    category = next((c for c, words in KEYWORDS.items() if any(w in t for w in words)), "endpoint")
    if any(w in t for w in ("everyone", "nobody", "whole", "hundreds", "all ")):
        priority = "P1"
    elif any(w in t for w in ("when convenient", "next week", "next month", "how to", "can you tell")):
        priority = "P4"
    else:
        priority = "P3"
    return {"category": category, "priority": priority, "assignment_group": ROUTING[category]}


baseline = pd.DataFrame([{"ticket_id": r.ticket_id, **keyword_baseline(r.text)} for r in TEST.itertuples()])
BASELINE_SCORE = score(baseline)
print("keyword baseline:", BASELINE_SCORE)

keyword baseline: {'category': 0.958, 'priority': 0.333, 'assignment_group': 0.875, 'all three': 0.25}


In [5]:
# (notebook) Check: the scorer and the label contract behave.
def check_part1():
    perfect = TEST[["ticket_id", *FIELDS]]
    assert score(perfect) == {"category": 1.0, "priority": 1.0, "assignment_group": 1.0, "all three": 1.0}
    assert 0 < BASELINE_SCORE["all three"] < 1
    try:
        TriageLabel(category="network", priority="P5", assignment_group="Network Ops")
    except ValidationError:
        return None
    raise AssertionError("TriageLabel must reject an unknown priority")


check("Part 1 — labelled set and scorer", check_part1)

✅ Part 1 — labelled set and scorer


## Part 2 — Zero-shot vs few-shot vs chain-of-thought, on both providers (20 min)

Three prompt templates share one role and one short label guide; they differ only in the
technique. Every call uses structured output (Part 4 explains it) so that parsing never blurs the
comparison. 3 techniques × 24 tickets × 2 providers = 144 calls, run 8 at a time.

In [6]:
# (notebook) Prompt templates: named, versioned and rendered in one place.
from dataclasses import dataclass
from string import Template


@dataclass(frozen=True)
class PromptTemplate:
    name: str
    version: str
    system: str
    user: str = "<ticket>\n$ticket\n</ticket>"
    schema: type[BaseModel] = TriageLabel

    def render(self, ticket: str) -> tuple[str, str]:
        """Return (system prompt, user message) for one ticket."""
        return self.system, Template(self.user).substitute(ticket=ticket)


class TriageWithReasoning(BaseModel):
    reasoning: str = Field(description="Brief step-by-step reasoning, written BEFORE the labels")
    category: Category
    priority: Priority
    assignment_group: Group


ROLE = "You are an L1 service-desk analyst at a large enterprise. Triage the IT ticket inside <ticket> tags."
SHORT_GUIDE = (
    "Categories: access (accounts, passwords, MFA, permissions) · network (VPN, Wi-Fi, LAN) · endpoint (laptops, "
    "printers, peripherals, software on a device) · email_collab (email, calendars, collaboration tools) · "
    "business_app (ERP and finance applications) · platform (servers, storage, VMs, pipelines) · security "
    "(phishing, malware, lost devices).\n"
    "Priorities: P1 critical · P2 high · P3 medium · P4 low.\n"
    "Assignment groups: Service Desk · Network Ops · Finance · Platform Engineering · Security Operations."
)


def examples_block(frame: pd.DataFrame, *, with_reason: bool = False) -> str:
    """Render labelled examples as XML-tagged few-shot demonstrations."""
    items = []
    for r in frame.itertuples():
        label = json.dumps({f: getattr(r, f) for f in FIELDS})
        reason = f"\n<why>{r.rationale}</why>" if with_reason else ""
        items.append(f"<example>\n<ticket>{r.text}</ticket>\n<label>{label}</label>{reason}\n</example>")
    return "<examples>\n" + "\n".join(items) + "\n</examples>"


ZERO_SHOT = PromptTemplate("zero-shot", "v1", f"{ROLE}\n\n{SHORT_GUIDE}")
FEW_SHOT = PromptTemplate("few-shot", "v1", f"{ROLE}\n\n{SHORT_GUIDE}\n\nLabelled examples from this service desk:\n{examples_block(EXAMPLES)}")
CHAIN_OF_THOUGHT = PromptTemplate(
    "chain-of-thought", "v1",
    f"{ROLE}\n\n{SHORT_GUIDE}\n\nThink before you label. Step 1: who or what is affected, and how many people? "
    "Step 2: has work stopped, is it degraded, or is this a request? Step 3: is there a security aspect? "
    "Write the steps briefly in `reasoning`, then give the labels.",
    schema=TriageWithReasoning,
)
V1 = [ZERO_SHOT, FEW_SHOT, CHAIN_OF_THOUGHT]

system, user = FEW_SHOT.render(TEST.text[0])
print(system[:900], "…\n")
print(user)

You are an L1 service-desk analyst at a large enterprise. Triage the IT ticket inside <ticket> tags.

Categories: access (accounts, passwords, MFA, permissions) · network (VPN, Wi-Fi, LAN) · endpoint (laptops, printers, peripherals, software on a device) · email_collab (email, calendars, collaboration tools) · business_app (ERP and finance applications) · platform (servers, storage, VMs, pipelines) · security (phishing, malware, lost devices).
Priorities: P1 critical · P2 high · P3 medium · P4 low.
Assignment groups: Service Desk · Network Ops · Finance · Platform Engineering · Security Operations.

Labelled examples from this service desk:
<examples>
<example>
<ticket>Since changing my password this morning the VPN client keeps rejecting my credentials. Webmail works fine with the new password. I'm working from home and can't reach any internal systems.</ticket>
<label>{"category": "net …

<ticket>
My account is locked after I mistyped my password a few times. I need to log in for a c

In [7]:
# (notebook) One classification call, on either provider, with the template's schema enforced.
def classify(provider: str, template: PromptTemplate, ticket_id: str, text: str) -> dict:
    system, user = template.render(text)
    row = {"provider": provider, "technique": template.name, "version": template.version, "ticket_id": ticket_id}
    started = time.perf_counter()
    try:
        if provider == "openai":
            r = openai_client.responses.parse(
                model=OPENAI_MODEL, instructions=system, input=user, text_format=template.schema,
                reasoning={"effort": "low"}, max_output_tokens=4_000,
            )
            label, tokens = r.output_parsed, (r.usage.input_tokens, r.usage.output_tokens)
        else:
            m = claude_client.messages.parse(
                model=ANTHROPIC_MODEL, max_tokens=4_000, system=system, output_config={"effort": "low"},
                output_format=template.schema, messages=[{"role": "user", "content": user}],
            )
            label, tokens = m.parsed_output, (m.usage.input_tokens, m.usage.output_tokens)
        row |= {f: getattr(label, f) for f in FIELDS}
        row |= {"reasoning": getattr(label, "reasoning", None), "input_tokens": tokens[0], "output_tokens": tokens[1]}
    except Exception as exc:  # a failed call is a data point, not a crash
        row["error"] = f"{exc.__class__.__name__}: {exc}"[:200]
    row["seconds"] = round(time.perf_counter() - started, 2)
    return row


def summarise(runs: pd.DataFrame) -> pd.DataFrame:
    """One row per provider × technique × version: accuracy, errors, tokens and latency."""
    rows = [{"provider": "—", "technique": "keyword baseline", "version": "—", **BASELINE_SCORE}]
    for (provider, technique, version), g in runs.groupby(["provider", "technique", "version"], sort=False):
        errors = int(g["error"].notna().sum()) if "error" in g else 0
        rows.append({"provider": provider, "technique": technique, "version": version, **score(g), "errors": errors,
                     "avg output tokens": round(g.output_tokens.mean()), "avg s": round(g.seconds.mean(), 1)})
    return pd.DataFrame(rows)

In [8]:
# (notebook) Run 3 techniques × 24 tickets on every configured provider.
jobs = [(p, t, r.ticket_id, r.text) for p in PROVIDERS for t in V1 for r in TEST.itertuples()]
started = time.perf_counter()
runs_v1 = pd.DataFrame(run_parallel(classify, jobs)) if jobs else pd.DataFrame()
print(f"{len(jobs)} calls in {time.perf_counter() - started:.0f}s")
summarise(runs_v1) if not runs_v1.empty else print("⚠️  No provider configured: skipping the comparison.")

144 calls in 34s


,provider,technique,version,category,priority,assignment_group,all three,errors,avg output tokens,avg s
0,—,keyword baseline,—,0.958,0.333,0.875,0.250,NaN,NaN,NaN
1,openai,zero-shot,v1,1.000,0.375,0.875,0.333,0.0,121.0,2.2
2,openai,few-shot,v1,1.000,0.625,0.958,0.625,0.0,110.0,1.9
3,openai,chain-of-thought,v1,1.000,0.375,0.958,0.375,0.0,226.0,2.7
4,anthropic,zero-shot,v1,1.000,0.583,0.958,0.542,0.0,32.0,1.3
5,anthropic,few-shot,v1,1.000,0.625,0.958,0.583,0.0,34.0,1.3
6,anthropic,chain-of-thought,v1,1.000,0.583,0.958,0.583,0.0,186.0,1.8


In [9]:
# (notebook) Check: every ticket got a valid label from every technique on every provider.
def check_part2():
    if runs_v1.empty:
        return "no provider configured"
    assert len(runs_v1) == len(PROVIDERS) * len(V1) * len(TEST)
    failed = runs_v1["error"].notna().mean() if "error" in runs_v1 else 0.0
    assert failed <= 0.1, f"{failed:.0%} of calls failed: look at runs_v1['error']"
    assert runs_v1.dropna(subset=["priority"]).priority.isin(["P1", "P2", "P3", "P4"]).all()


check("Part 2 — technique comparison", check_part2)

✅ Part 2 — technique comparison


📝 **Record (Part 2).** Copy your numbers; they will differ from the guide's.

| Provider | Technique | All three correct | Priority | Avg output tokens | Avg seconds |
| -------- | --------- | ----------------- | -------- | ----------------- | ----------- |
| openai | zero-shot | | | | |
| openai | few-shot | | | | |
| openai | chain-of-thought | | | | |
| anthropic | zero-shot | | | | |
| anthropic | few-shot | | | | |
| anthropic | chain-of-thought | | | | |

Which technique helped most, and on which field? What did chain-of-thought cost in output tokens?

## Part 3 — Debug the prompt, then price it (15 min)

Look at the errors before you change anything. Find the tickets that most runs got wrong, read
the gold rationale, and decide whether the cause is the technique or missing instructions.

In [ ]:
# (notebook) Error analysis: which tickets do most v1 runs get wrong, and on which field?
def wrong_fields(runs: pd.DataFrame) -> pd.DataFrame:
    merged = TEST.merge(runs, on="ticket_id", suffixes=("", "_pred"))
    for f in FIELDS:
        merged[f"{f}_wrong"] = merged[f] != merged[f + "_pred"]
    return merged


if not runs_v1.empty:
    wf = wrong_fields(runs_v1)
    runs_per_ticket = len(PROVIDERS) * len(V1)
    hardest = (
        wf.groupby("ticket_id")[[f"{f}_wrong" for f in FIELDS]].sum()
        .assign(total=lambda d: d.sum(axis=1)).sort_values("total", ascending=False).head(6)
        .join(TEST.set_index("ticket_id")[["text", "priority", "rationale"]])
    )
    print(f"Wrong-label counts out of {runs_per_ticket} runs per ticket:")
    display(hardest)
    display(pd.crosstab(wf.priority, wf.priority_pred, rownames=["gold"], colnames=["predicted"]))

In [ ]:
# (notebook) v2: the fix the error analysis points to. Give the model the policy it is graded against.
ZERO_SHOT_V2 = PromptTemplate("zero-shot", "v2", f"{ROLE}\n\nApply this policy exactly:\n<policy>\n{POLICY}\n</policy>")
FEW_SHOT_V2 = PromptTemplate(
    "few-shot", "v2",
    f"{ZERO_SHOT_V2.system}\n\nLabelled examples, with the reason for each label:\n{examples_block(EXAMPLES, with_reason=True)}",
)
V2 = [ZERO_SHOT_V2, FEW_SHOT_V2]

jobs = [(p, t, r.ticket_id, r.text) for p in PROVIDERS for t in V2 for r in TEST.itertuples()]
runs_v2 = pd.DataFrame(run_parallel(classify, jobs)) if jobs else pd.DataFrame()
if not runs_v2.empty:
    display(summarise(pd.concat([runs_v1, runs_v2])))

In [ ]:
# (notebook) Regression check: a fix that breaks other tickets is not a fix.
def regressions(before: pd.DataFrame, after: pd.DataFrame) -> pd.DataFrame:
    """Tickets that were fully right before and are wrong after, per provider and technique."""
    keys = ["provider", "technique", "ticket_id"]
    b = wrong_fields(before).assign(ok=lambda d: ~d[[f"{f}_wrong" for f in FIELDS]].any(axis=1))[keys + ["ok"]]
    a = wrong_fields(after).assign(ok=lambda d: ~d[[f"{f}_wrong" for f in FIELDS]].any(axis=1))[keys + ["ok"]]
    both = b.merge(a, on=keys, suffixes=("_v1", "_v2"))
    return both[both.ok_v1 & ~both.ok_v2]


if not runs_v2.empty:
    broken = regressions(runs_v1, runs_v2)
    print(f"{len(broken)} regression(s) from v1 to v2")
    display(broken)

In [ ]:
# (notebook) Your turn (optional): a v3 template that fixes a miss you still see.
# TODO (optional): set MY_TEMPLATE, e.g.
# MY_TEMPLATE = PromptTemplate("few-shot", "v3", FEW_SHOT_V2.system + "\n\nAlso: ...")
MY_TEMPLATE = None
if MY_TEMPLATE:
    jobs = [(p, MY_TEMPLATE, r.ticket_id, r.text) for p in PROVIDERS for r in TEST.itertuples()]
    display(summarise(pd.DataFrame(run_parallel(classify, jobs))))

In [ ]:
# (notebook) Price each template: prompt tokens per ticket, counted by each provider's own endpoint.
def prompt_tokens(provider: str, template: PromptTemplate) -> int:
    system, user = template.render(TEST.text[0])
    if provider == "openai":
        return openai_client.responses.input_tokens.count(model=OPENAI_MODEL, instructions=system, input=user).input_tokens
    return claude_client.messages.count_tokens(
        model=ANTHROPIC_MODEL, system=system, messages=[{"role": "user", "content": user}]
    ).input_tokens


if PROVIDERS:
    token_table = pd.DataFrame(
        [{"template": f"{t.name} {t.version}", **{p: prompt_tokens(p, t) for p in PROVIDERS}} for t in V1 + V2]
    ).set_index("template")
    display(token_table)

In [ ]:
# (notebook) Prompt caching: the few-shot v2 system prompt is identical for every ticket, so cache it.
def cached_call(provider: str, template: PromptTemplate, text: str) -> dict:
    system, user = template.render(text)
    if provider == "openai":  # automatic for prompts of 1,024+ tokens; the key routes requests to the same cache
        r = openai_client.responses.parse(
            model=OPENAI_MODEL, instructions=system, input=user, text_format=template.schema,
            reasoning={"effort": "low"}, max_output_tokens=4_000, prompt_cache_key="lab01-few-shot-v2",
        )
        u = r.usage
        return {"provider": provider, "input total": u.input_tokens, "read from cache": u.input_tokens_details.cached_tokens}
    m = claude_client.messages.parse(  # explicit breakpoint on the system block
        model=ANTHROPIC_MODEL, max_tokens=4_000, output_config={"effort": "low"}, output_format=template.schema,
        system=[{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user}],
    )
    u = m.usage
    return {"provider": provider, "input total": u.input_tokens + u.cache_creation_input_tokens + u.cache_read_input_tokens,
            "written to cache": u.cache_creation_input_tokens, "read from cache": u.cache_read_input_tokens}


cache_rows = []
for provider in PROVIDERS:
    for i, text in enumerate(TEST.text[:4]):
        cache_rows.append({"call": i + 1, **cached_call(provider, FEW_SHOT_V2, text)})
        time.sleep(2)  # a cache entry becomes readable shortly after the request that wrote it
pd.DataFrame(cache_rows).fillna(0) if cache_rows else None

In [ ]:
# (notebook) Check: v2 ran, and you priced the templates.
def check_part3():
    if runs_v2.empty:
        return "no provider configured"
    assert set(runs_v2.version) == {"v2"} and len(runs_v2) == len(PROVIDERS) * len(V2) * len(TEST)
    assert (token_table.loc["few-shot v2"] > token_table.loc["zero-shot v1"]).all(), "few-shot v2 should cost more tokens"


check("Part 3 — debugging and cost", check_part3)

## Part 4 — Enforce a JSON schema on document extraction (15 min)

Change requests arrive as emails in different shapes and languages. The change-management
database needs the same fields every time. Compare asking for JSON in the prompt with having
the API enforce the schema.

In [ ]:
# (notebook) Five change-request documents, their gold extraction, and the target schema.
from datetime import datetime, timedelta

DOCS = {p.stem: p.read_text(encoding="utf-8") for p in sorted((DATA_DIR / "change_requests").glob("CHG-*.txt"))}
GOLD = {k: v for k, v in json.loads((DATA_DIR / "change_requests" / "ground_truth.json").read_text()).items() if k.startswith("CHG-")}


class ChangeRequest(BaseModel):
    change_id: str = Field(description="Change ID exactly as written, e.g. CHG-20931")
    title: str = Field(description="Short title of the change, in English")
    change_type: Literal["standard", "normal", "emergency"]
    risk: Literal["low", "medium", "high"] = Field(description="Risk level as stated in the document")
    requested_by_team: Group
    region: Literal["EU", "US", "APAC"]
    affected_services: list[str] = Field(description="Host names or CI IDs the change touches, lower case")
    window_start: datetime = Field(description="Start of the CURRENT implementation window, ISO 8601 with UTC offset")
    window_end: datetime = Field(description="End of the current implementation window, ISO 8601 with UTC offset")
    cab_required: bool = Field(description="True if the document says CAB approval is needed")
    rollback_plan: str | None = Field(description="The rollback or back-out plan in one sentence, or null if the document has none")


INSTRUCTIONS = (
    "You extract change requests for the change-management database. If the document reschedules the change, "
    "use the current window. Use null for anything the document does not state; never invent a value."
)
print(DOCS["CHG-20934"])

In [ ]:
# (notebook) Approach A: ask for JSON in the prompt, then parse and validate it yourself.
PROMPT_ONLY_SYSTEM = (
    f"{INSTRUCTIONS}\nReturn ONLY a JSON object that matches this JSON Schema:\n"
    + json.dumps(ChangeRequest.model_json_schema())
)


def parse_reply(text: str) -> tuple[str, ChangeRequest | None]:
    """Outcome of json.loads + schema validation on a raw model reply."""
    try:
        data = json.loads(text)
    except json.JSONDecodeError as exc:
        return f"invalid JSON ({exc.msg})", None
    try:
        return "valid", ChangeRequest.model_validate(data)
    except ValidationError as exc:
        first = exc.errors()[0]
        return f"schema error ×{exc.error_count()}: {'.'.join(map(str, first['loc']))}: {first['msg']}", None


def extract_prompt_only(provider: str, doc_id: str) -> dict:
    if provider == "openai":
        r = openai_client.responses.create(
            model=OPENAI_MODEL, instructions=PROMPT_ONLY_SYSTEM, input=DOCS[doc_id],
            reasoning={"effort": "low"}, max_output_tokens=4_000,
        )
        text = r.output_text
    else:
        m = claude_client.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=4_000, system=PROMPT_ONLY_SYSTEM, output_config={"effort": "low"},
            messages=[{"role": "user", "content": DOCS[doc_id]}],
        )
        text = claude_text(m)
    outcome, _ = parse_reply(text)
    return {"provider": provider, "approach": "A: JSON in the prompt", "doc": doc_id, "outcome": outcome, "starts with": text[:25]}


prompt_only = pd.DataFrame(run_parallel(extract_prompt_only, [(p, d) for p in PROVIDERS for d in DOCS]))
prompt_only

In [ ]:
# (notebook) Deliberate failures. First, a reply recorded from an earlier run: valid content, invalid JSON.
RECORDED_REPLY = (
    "Here is the extracted change request:\n```json\n"
    '{"change_id": "CHG-20932", "change_type": "standard", "risk": "low"}\n'
    "```\nLet me know if you need anything else."
)
print("recorded reply →", parse_reply(RECORDED_REPLY)[0])

# Second, the old trick of pre-filling the assistant turn with "{" to force JSON.
if claude_client:
    try:
        claude_client.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=500, system=PROMPT_ONLY_SYSTEM,
            messages=[{"role": "user", "content": DOCS["CHG-20932"]}, {"role": "assistant", "content": "{"}],
        )
        print(f"ℹ️  {ANTHROPIC_MODEL} still accepts a prefilled assistant turn.")
    except anthropic.BadRequestError as exc:
        print(f"❌ {ANTHROPIC_MODEL} → 400 BadRequestError: {exc.body['error']['message'] if isinstance(exc.body, dict) else exc}")

In [ ]:
# (notebook) Approach B: the API enforces the schema (constrained decoding), and the SDK parses into your model.
def extract_native(provider: str, messages: list[dict]) -> ChangeRequest:
    """One structured-output call; `messages` is the conversation so far (user turn last)."""
    if provider == "openai":
        r = openai_client.responses.parse(
            model=OPENAI_MODEL, instructions=INSTRUCTIONS, input=messages, text_format=ChangeRequest,
            reasoning={"effort": "low"}, max_output_tokens=4_000,
        )
        return r.output_parsed
    m = claude_client.messages.parse(
        model=ANTHROPIC_MODEL, max_tokens=4_000, system=INSTRUCTIONS, output_config={"effort": "low"},
        output_format=ChangeRequest, messages=messages,
    )
    return m.parsed_output


def native_row(provider: str, doc_id: str) -> dict:
    try:
        cr = extract_native(provider, [{"role": "user", "content": DOCS[doc_id]}])
        return {"provider": provider, "approach": "B: schema enforced", "doc": doc_id, "outcome": "valid", "record": cr}
    except Exception as exc:
        return {"provider": provider, "approach": "B: schema enforced", "doc": doc_id, "outcome": f"{exc.__class__.__name__}"}


native = pd.DataFrame(run_parallel(native_row, [(p, d) for p in PROVIDERS for d in DOCS]))
if not native.empty:
    both = pd.concat([prompt_only, native])
    display(both.assign(valid=both.outcome.eq("valid")).pivot_table(index="provider", columns="approach", values="valid", aggfunc="sum"))

In [ ]:
# (notebook) Check: approach B returned a validated ChangeRequest for every document.
def check_part4():
    if native.empty:
        return "no provider configured"
    assert native.outcome.eq("valid").all(), native[native.outcome.ne("valid")]
    assert all(isinstance(r, ChangeRequest) for r in native.record)


check("Part 4 — schema enforcement", check_part4)

## Part 5 — Validate the output, retry with feedback, route what is left (15 min)

A schema guarantees shape, not truth. Add the rules the JSON Schema cannot express, feed
validation errors back once, and send what still fails, or what breaks a business rule, to a person.

In [ ]:
# (notebook) Layer 1: format rules the schema did not express. Layer 2: business rules.
import re

from pydantic import field_validator, model_validator


class CheckedChangeRequest(ChangeRequest):
    @field_validator("change_id")
    @classmethod
    def id_format(cls, value: str) -> str:
        if not re.fullmatch(r"CHG-\d{5}", value):
            raise ValueError("change_id must look like CHG-12345")
        return value

    @field_validator("window_start", "window_end")
    @classmethod
    def has_offset(cls, value: datetime) -> datetime:
        if value.tzinfo is None:
            raise ValueError("timestamps need a UTC offset, e.g. 2026-11-07T22:00:00+01:00")
        return value

    @model_validator(mode="after")
    def window_is_sane(self):
        if self.window_end <= self.window_start:
            raise ValueError("window_end must be after window_start")
        if self.window_end - self.window_start > timedelta(hours=12):
            raise ValueError("windows longer than 12 hours need a separate review")
        return self


def business_rule_violations(cr: ChangeRequest) -> list[str]:
    """Rules about the change itself: a person decides, the model cannot fix them."""
    found = []
    if cr.risk == "high" and not cr.cab_required:
        found.append("high-risk change without CAB approval")
    if cr.change_type == "standard" and cr.risk != "low":
        found.append("a standard (pre-approved) change must be low risk")
    if cr.change_type != "standard" and cr.rollback_plan is None:
        found.append("normal and emergency changes need a rollback plan")
    return found


reversed_window = {**GOLD["CHG-20931"], "title": "test", "rollback_plan": "re-flash",
                   "window_start": "2026-11-08T02:00:00+01:00", "window_end": "2026-11-07T22:00:00+01:00"}
try:
    CheckedChangeRequest.model_validate(reversed_window)
except ValidationError as exc:
    print("layer 1 →", exc.errors()[0]["msg"])
gold_20935 = ChangeRequest.model_validate({**GOLD["CHG-20935"], "title": "test", "rollback_plan": "vendor downgrade"})
print("layer 2 →", business_rule_violations(gold_20935))

In [ ]:
# (notebook) Extract → validate → one retry with the errors → route.
def extract_with_retry(provider: str, doc_id: str, max_attempts: int = 2) -> dict:
    messages = [{"role": "user", "content": DOCS[doc_id]}]
    errors: list[str] = []
    for attempt in range(1, max_attempts + 1):
        cr = extract_native(provider, messages)
        try:
            checked = CheckedChangeRequest.model_validate(cr.model_dump())
        except ValidationError as exc:
            errors = [f"{'.'.join(map(str, e['loc'])) or 'record'}: {e['msg']}" for e in exc.errors()]
            messages += [
                {"role": "assistant", "content": cr.model_dump_json()},
                {"role": "user", "content": "That extraction failed validation:\n- " + "\n- ".join(errors)
                 + "\nRe-read the document and return a corrected extraction."},
            ]
            continue
        violations = business_rule_violations(checked)
        route = "human review: " + "; ".join(violations) if violations else "auto-accept"
        return {"provider": provider, "doc": doc_id, "attempts": attempt, "route": route, "record": checked}
    return {"provider": provider, "doc": doc_id, "attempts": max_attempts,
            "route": "human review: failed validation twice", "errors": errors, "record": None}


def field_check(cr: ChangeRequest, gold: dict) -> dict:
    """Which of 8 scored fields match the gold extraction (windows compare as instants)."""
    return {
        "change_type": cr.change_type == gold["change_type"], "risk": cr.risk == gold["risk"],
        "team": cr.requested_by_team == gold["requested_by_team"], "region": cr.region == gold["region"],
        "start": cr.window_start == datetime.fromisoformat(gold["window_start"]),
        "end": cr.window_end == datetime.fromisoformat(gold["window_end"]),
        "cab": cr.cab_required == gold["cab_required"],
        "rollback": (cr.rollback_plan is not None) == gold["rollback_plan_present"],
    }


routed = run_parallel(extract_with_retry, [(p, d) for p in PROVIDERS for d in DOCS])
rows = []
for r in routed:
    fields = field_check(r["record"], GOLD[r["doc"]]) if r["record"] else {}
    rows.append({"provider": r["provider"], "doc": r["doc"], "attempts": r["attempts"], "route": r["route"],
                 "fields correct": f"{sum(fields.values())}/8" if fields else "—",
                 "wrong": ", ".join(k for k, ok in fields.items() if not ok)})
pd.DataFrame(rows)

In [ ]:
# (notebook) Check: the validators work and every document has a route.
def check_part5():
    try:
        CheckedChangeRequest.model_validate(reversed_window)
    except ValidationError:
        pass
    else:
        raise AssertionError("a reversed window must fail validation")
    assert business_rule_violations(gold_20935) == ["high-risk change without CAB approval"]
    if not PROVIDERS:
        return "no provider configured (validators checked offline)"
    assert all(r["route"] for r in routed) and len(routed) == len(PROVIDERS) * len(DOCS)
    for r in routed:
        if r["doc"] == "CHG-20935" and r["route"] == "auto-accept":
            print(f"⚠️  {r['provider']} auto-accepted CHG-20935: look at its cab_required value. That is a finding.")


check("Part 5 — validation and routing", check_part5)

📝 **Write-up (3–5 sentences).** Which prompting technique would you ship for triage, on which
provider, and why? What did structured output fix, and what did it not fix? Which of the five
change requests should a person see, and which rule sent it there?

> *Your answer here.*

---

**Done.** Answer the checkpoint questions in the lab guide, then go on to
[Lab 02 — One assistant, two SDKs](../lab-02-one-assistant-two-sdks.md).